# Qwen2.5-Coder 3B on Kaggle Notebooks

This is an optional notebook-local inference fallback for the Colab workflow. It does not expose a network service or connect to the workspace task board. Kaggle accelerator availability, weekly usage allowances, session limits, and storage policies can change; check the current Kaggle account and notebook UI rather than relying on a fixed GPU-hours figure.

**Recommended setup:** Create a Kaggle Dataset containing the Hugging Face snapshot for `Qwen/Qwen2.5-Coder-3B-Instruct`, then attach it with **Add Input**. Select one GPU (T4 or P100 when available) and enable Internet if dependencies still need installation. This avoids downloading the multi-gigabyte model during each inference session. Do not select two GPUs for this 3B 4-bit model; the measured T4 run used about 2 GiB peak allocated VRAM, and this notebook explicitly places the model on the first GPU.

The attached dataset is read-only and persists independently of the session. `/kaggle/working` is session output storage, not a substitute for an attached persistent model dataset. Start a session only when ready, run setup and inference as needed, and use Kaggle's **Stop Session** control when finished. Do not expect a notebook background commit to preserve a running interactive GPU session.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a Kaggle GPU accelerator and start a new session before loading the model.")
for index in range(torch.cuda.device_count()):
    gpu = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {gpu.name} ({gpu.total_memory / 1024**3:.1f} GiB VRAM)")
print(f"CUDA: {torch.version.cuda}")

## Attach the model dataset

Use **Add Input** to attach the dataset containing the model snapshot. Set `MODEL_DIR` below to the mounted dataset directory (Kaggle typically mounts inputs under `/kaggle/input/<dataset-slug>`). Keep the snapshot files together at that directory level. If the path is wrong, list `/kaggle/input` and update it; do not download the model again unless you intentionally want to spend session time and bandwidth on that transfer.

In [ ]:
from pathlib import Path
import os

MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
MODEL_DIR = Path("/kaggle/input/qwen25-coder-3b-instruct")
HF_HOME = Path("/kaggle/working/hf_cache")
PIP_CACHE_DIR = Path("/kaggle/working/pip-cache")
if not MODEL_DIR.is_dir():
    available_inputs = sorted(path.name for path in Path("/kaggle/input").iterdir())
    raise FileNotFoundError(f"Model dataset not found at {MODEL_DIR}. Attached inputs: {available_inputs}. Update MODEL_DIR to the attached dataset path.")
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["PIP_CACHE_DIR"] = str(PIP_CACHE_DIR)
print(f"Model snapshot: {MODEL_DIR}")
print("HF and pip caches are session-local under /kaggle/working.")

## Install inference dependencies

Kaggle's preinstalled PyTorch is reused. This installs only the Hugging Face and quantization packages required by the model into the current session; the pip cache under `/kaggle/working` is not assumed to survive a new session. If the required compatible packages are already available, skip this cell.

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "--cache-dir", str(PIP_CACHE_DIR), "transformers>=4.45,<5", "accelerate", "bitsandbytes"])
print(f"Using PyTorch {torch.__version__}.")

## Load the model and run inference

The model uses 4-bit NF4 weights, FP16 compute, SDPA attention, and the KV cache. `device_map={"": 0}` keeps it on the first GPU even when a multi-GPU accelerator is selected; one GPU is sufficient for this model based on the prior T4 measurement. The one-token warm-up keeps the benchmark comparable with the Colab notebook and isolates steady-state generation speed; it is not a reduction in total work for a single request.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), quantization_config=quantization, device_map={"": 0}, torch_dtype=torch.float16, attn_implementation="sdpa", local_files_only=True)
model.eval()
print(f"Loaded {MODEL_ID} in 4-bit mode on GPU 0 ({torch.cuda.get_device_name(0)}).")

In [ ]:
import json
import time

task = {"title": "Add validation to the task creation API", "project": "api", "task_type": "feature", "description": "Reject invalid task input and return clear errors.", "acceptance_criteria": "Invalid titles and unsupported fields return 400; valid tasks still persist."}
messages = [{"role": "system", "content": "Refine this software task. Return only a JSON object with string fields description and acceptance_criteria. The acceptance_criteria value must be one string, never an array or object. Preserve intent; do not invent requirements or claim work is done. Make criteria concise and verifiable."}, {"role": "user", "content": json.dumps(task, ensure_ascii=False)}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda:0")
if inputs["input_ids"].shape[-1] > 2048: raise ValueError("Prompt exceeds the 2048-token evaluation budget; shorten the task fields.")
with torch.inference_mode(): warmup_output = model.generate(**inputs, max_new_tokens=1, do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
torch.cuda.synchronize()
del warmup_output
torch.cuda.reset_peak_memory_stats(0)
started = time.perf_counter()
with torch.inference_mode(): output = model.generate(**inputs, max_new_tokens=220, do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
torch.cuda.synchronize()
elapsed = time.perf_counter() - started
peak_gpu_memory_mb = torch.cuda.max_memory_allocated(0) / 1024**2
peak_gpu_reserved_mb = torch.cuda.max_memory_reserved(0) / 1024**2
generated_tokens = output.shape[-1] - inputs["input_ids"].shape[-1]
answer = tokenizer.decode(output[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
print(f"Elapsed: {elapsed:.2f}s | Generated tokens: {generated_tokens} | Speed: {generated_tokens / max(elapsed, 1e-6):.2f} tokens/s | Peak allocated VRAM: {peak_gpu_memory_mb:.0f} MiB | Peak reserved VRAM: {peak_gpu_reserved_mb:.0f} MiB")
print(answer)

In [ ]:
try:
    json_answer = answer.strip()
    if json_answer.startswith("```"):
        json_answer = json_answer.removeprefix("```json").removeprefix("```")
        json_answer = json_answer.removesuffix("```").strip()
    draft = json.loads(json_answer)
    if not isinstance(draft, dict):
        raise ValueError("The model response must be a JSON object")
    criteria = draft.get("acceptance_criteria")
    if isinstance(criteria, list):
        formatted = []
        for index, item in enumerate(criteria, start=1):
            if isinstance(item, str):
                details = item.strip()
            elif isinstance(item, dict) and all(isinstance(value, str) for value in item.values()):
                details = "; ".join(f"{field.replace('_', ' ')}: {value.strip()}" for field, value in item.items() if value.strip())
            else:
                raise ValueError("Invalid acceptance criterion")
            if not details:
                raise ValueError("Empty acceptance criterion")
            formatted.append(f"{index}. {details}")
        draft["acceptance_criteria"] = "\n".join(formatted)
    assert isinstance(draft.get("description"), str)
    assert isinstance(draft.get("acceptance_criteria"), str)
    print("Output check: valid JSON with description and acceptance_criteria.")
except (json.JSONDecodeError, AssertionError, ValueError):
    print("Output check: the model did not return the expected JSON shape; review or retry manually.")

## Stop GPU usage

When finished, stop the Kaggle session using the notebook's **Stop Session** control. This notebook does not try to stop the session from Python. Clearing objects below releases GPU memory but does not stop the accelerator or guarantee that the session has ended.

In [ ]:
import gc

for name in ("inputs", "output", "model", "tokenizer", "answer", "draft"):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Model objects cleared. Stop the Kaggle session using the notebook UI when finished.")